# Stacking Ensemble — Meta-Learner | Dengue Forecasting IMDC 2026

Pipeline final da modelagem:
1. **Comparação lags vs sem-lags** (clássicos e MLP) → escolhe o melhor de cada
2. **Heatmap do melhor modelo** por UF × Validation Test
3. **Stacking** com meta-learner combinando os modelos-base vencedores

O meta-learner é treinado **sobre as previsões** dos modelos-base (conteúdo dos parquets), aprendendo a combiná-las de forma ótima — Stacked Generalization (Breiman, 1996; Peres et al., 2022).

### 1. Imports & Paths

In [ ]:
import warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR = Path(r"C:\myenv\env\Mestrado\2026_imdc_challenge")
OUT_DIR  = BASE_DIR / "8_results_stacking"
OUT_DIR.mkdir(exist_ok=True)

PI_LEVELS  = [0.50, 0.80, 0.90, 0.95]
QUANTILES  = sorted(set(
    [0.5] +
    [(1-a)/2   for a in PI_LEVELS] +
    [1-(1-a)/2 for a in PI_LEVELS]
))  # [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]

# Mapeamento quantil → coluna no parquet
def q_to_col(q):
    if abs(q-0.5) < 1e-9: return "median"
    for a in PI_LEVELS:
        if abs(q-(1-a)/2)   < 1e-9: return f"lower_{int(a*100)}"
        if abs(q-(1-(1-a)/2)) < 1e-9: return f"upper_{int(a*100)}"
    return None

# Caminhos dos parquets
PATHS = {
    "classical_nolag": BASE_DIR / "3_results_classical"    / "forecasts_classical.parquet",
    "mlp_nolag"      : BASE_DIR / "4_results_mlp"          / "forecasts_mlp.parquet",
    "classical_lag"  : BASE_DIR / "5_results_classical_v2" / "forecasts_classical.parquet",
    "mlp_lag"        : BASE_DIR / "6_results_mlp_v2"       / "forecasts_mlp.parquet",
    "esn"            : BASE_DIR / "7_results_esn"          / "forecasts_esn.parquet",
}


### 2. WIS & Loaders

In [60]:
def wis_row(y_true, median, lowers, uppers, levels=PI_LEVELS):
    """WIS de um único ponto. lowers/uppers: dict {alpha: valor}."""
    K = len(levels)
    s = 0.5 * abs(y_true - median)
    for a in levels:
        lo, hi = lowers[a], uppers[a]
        IS = (hi - lo
              + (2/a)*max(0, lo - y_true)
              + (2/a)*max(0, y_true - hi))
        s += (a/2)*IS
    return s / (K + 0.5)

def add_wis_column(df):
    """Adiciona coluna WIS linha-a-linha."""
    lo = {a: df[f"lower_{int(a*100)}"].values for a in PI_LEVELS}
    hi = {a: df[f"upper_{int(a*100)}"].values for a in PI_LEVELS}
    yt = df["y_true"].values; md = df["median"].values
    out = np.zeros(len(df))
    for i in range(len(df)):
        out[i] = wis_row(yt[i], md[i],
                         {a: lo[a][i] for a in PI_LEVELS},
                         {a: hi[a][i] for a in PI_LEVELS})
    df = df.copy(); df["WIS"] = out
    return df

def load_forecasts(path, model_name):
    df = pd.read_parquet(path)
    df["date"] = pd.to_datetime(df["date"])
    df = add_wis_column(df)
    df["source"] = model_name
    return df

dfs = {}
for name, p in PATHS.items():
    if p.exists():
        dfs[name] = load_forecasts(p, name)
        print(f"  {name:18s} → {len(dfs[name]):>6,} linhas | "
              f"WIS médio = {dfs[name]['WIS'].mean():.2f}")
    else:
        print(f"  {name:18s} → AUSENTE ({p})")


  classical_nolag    → 13,962 linhas | WIS médio = inf
  mlp_nolag          →  4,654 linhas | WIS médio = 41418038.16
  classical_lag      → 13,962 linhas | WIS médio = 2836879707746867587208528088677276514560815994405367840868663296.00
  mlp_lag            →  4,654 linhas | WIS médio = 740328.58
  esn                →  4,654 linhas | WIS médio = 246284660863369696.00


In [61]:
# ═══════════════════════════════════════════════════════════════════════════
# SANEAMENTO NUMÉRICO DOS MODELOS-BASE
# ───────────────────────────────────────────────────────────────────────────
# Os três modelos apresentam previsões patológicas em estados específicos:
#   • Classical (SARIMAX): explosão a ±1e65 e valores NEGATIVOS (MA, RO, PA, RS, SP)
#   • MLP: explosão a 1e10 (SP)
#   • ESN: colapso a zero (PR, DF, RS, SP, SC) e explosão a 1e16 (AP, RR)
#
# Estratégia: detectar linhas inválidas (não-finitas, negativas, ou acima de um
# teto físico = K × pico histórico da UF) e substituí-las por uma previsão
# ingênua de climatologia (média sazonal por semana epidemiológica da UF).
# Isso garante previsões finitas e plausíveis sem mascarar a falha — o número
# de linhas saneadas é reportado e vira limitação documentada.
# ═══════════════════════════════════════════════════════════════════════════

CEILING_FACTOR = 50.0   # teto = 50 × pico histórico observado da UF

QCOLS = (["median"]
         + [f"lower_{int(a*100)}" for a in PI_LEVELS]
         + [f"upper_{int(a*100)}" for a in PI_LEVELS])

# Climatologia por UF×semana, a partir da série histórica completa
hist = pd.read_parquet(BASE_DIR / "0_databases" / "hierarch_forecast_lagged.parquet")
hist = hist[hist["uf"] != "ES"].copy()
hist["date"]  = pd.to_datetime(hist["date"])
hist["casos"] = hist["casos"].astype(float)
hist_uf = hist.groupby(["uf","date"])["casos"].sum().reset_index()
hist_uf["week"] = hist_uf["date"].dt.isocalendar().week.astype(int)
clim = hist_uf.groupby(["uf","week"])["casos"].mean()          # média sazonal
clim_uf_mean = hist_uf.groupby("uf")["casos"].mean()           # fallback geral
uf_peak = hist_uf.groupby("uf")["casos"].max()                 # pico histórico


def sanitize_forecasts(df, model_label):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["week"] = df["date"].dt.isocalendar().week.astype(int)

    # Teto por UF
    ceil_map = (uf_peak * CEILING_FACTOR).to_dict()
    ceil_arr = df["uf"].map(ceil_map).values

    # Máscara de linha patológica (qualquer coluna de quantil viola)
    bad = np.zeros(len(df), dtype=bool)
    for c in QCOLS:
        v = df[c].values
        bad |= ~np.isfinite(v)          # inf / nan
        bad |= (v < 0)                  # negativo (impossível p/ casos)
        bad |= (v > ceil_arr)           # acima do teto físico
    # Colapso total: todos os intervalos de largura zero E mediana zero
    collapse = np.ones(len(df), dtype=bool)
    for a in PI_LEVELS:
        collapse &= (df[f"lower_{int(a*100)}"].values
                     == df[f"upper_{int(a*100)}"].values)
    collapse &= (df["median"].values <= 1e-9)
    bad |= collapse

    n_bad = int(bad.sum())

    # Substituição por climatologia nas linhas ruins
    if n_bad > 0:
        idx = np.where(bad)[0]
        for i in idx:
            uf = df.iloc[i]["uf"]; wk = int(df.iloc[i]["week"])
            base = clim.get((uf, wk), clim_uf_mean.get(uf, 0.0))
            base = float(max(base, 0.0))
            # mediana = climatologia; intervalos proporcionais (incerteza ±)
            df.iat[i, df.columns.get_loc("median")] = base
            for a in PI_LEVELS:
                z = {0.5:0.67, 0.8:1.28, 0.9:1.64, 0.95:1.96}[a]
                half = z * max(base * 0.5, 1.0)   # dispersão ingênua
                df.iat[i, df.columns.get_loc(f"lower_{int(a*100)}")] = max(base-half, 0)
                df.iat[i, df.columns.get_loc(f"upper_{int(a*100)}")] = base+half

    # Relatório por UF
    df["_bad"] = bad
    by_uf = df[df["_bad"]].groupby("uf").size()
    print(f"\n[{model_label}] linhas saneadas: {n_bad:,}/{len(df):,} "
          f"({100*n_bad/len(df):.1f}%)")
    if n_bad > 0:
        print(f"  Por UF: {by_uf.to_dict()}")

    return df.drop(columns=["week","_bad"])


# Aplica aos três modelos-base ANTES do stacking
print("="*64)
print("SANEAMENTO DOS MODELOS-BASE")
print("="*64)
for src in [winner_classical, winner_mlp, "esn"]:
    if src in dfs:
        dfs[src] = add_wis_column(           # recalcula WIS pós-saneamento
            sanitize_forecasts(dfs[src], src)
        )

print("\n✓ Saneamento concluído. Os dfs já estão limpos para o stacking.")

SANEAMENTO DOS MODELOS-BASE

[classical_lag] linhas saneadas: 4,162/13,962 (29.8%)
  Por UF: {'AL': 20, 'AP': 227, 'CE': 17, 'DF': 108, 'MA': 537, 'MG': 10, 'PA': 515, 'PB': 463, 'PI': 3, 'PR': 50, 'RJ': 527, 'RN': 475, 'RO': 41, 'RR': 358, 'RS': 122, 'SC': 145, 'SE': 485, 'SP': 49, 'TO': 10}

[mlp_nolag] linhas saneadas: 225/4,654 (4.8%)
  Por UF: {'AL': 21, 'DF': 4, 'PR': 35, 'RR': 51, 'RS': 45, 'SC': 11, 'SP': 33, 'TO': 25}

[esn] linhas saneadas: 1,020/4,654 (21.9%)
  Por UF: {'AL': 51, 'AP': 81, 'CE': 96, 'DF': 131, 'PR': 137, 'RR': 176, 'RS': 127, 'SC': 105, 'SP': 116}

✓ Saneamento concluído. Os dfs já estão limpos para o stacking.


### 3. Comparação: Lags vs. Sem-Lags (Clássicos e MLP)

Compara o WIS médio das versões com e sem defasagem climática, escolhendo a melhor de cada família para entrar no stacking.

In [62]:
def mean_wis(df):
    """WIS mediano por (uf, validation) → média das medianas.
    A mediana por grupo é robusta aos valores explosivos do SARIMAX/ARIMA;
    a agregação externa evita que uma única UF divergente domine a comparação."""
    wis_por_grupo = df.groupby(["uf","validation"])["WIS"].median()
    return wis_por_grupo.median()

# Diagnóstico: compara agregação por média (sensível a outliers) vs mediana (robusta)
for src in ["classical_nolag","classical_lag"]:
    if src in dfs:
        d = dfs[src]
        print(f"{src:18s} | média={d['WIS'].mean():.2e} | "
              f"mediana_robusta={d.groupby(['uf','validation'])['WIS'].median().median():.2f}")

comparison = []
# Clássicos
if "classical_nolag" in dfs and "classical_lag" in dfs:
    w_nolag = mean_wis(dfs["classical_nolag"])
    w_lag   = mean_wis(dfs["classical_lag"])
    winner_classical = "classical_lag" if w_lag < w_nolag else "classical_nolag"
    comparison.append(("Clássicos", w_nolag, w_lag,
                       "COM lags" if w_lag < w_nolag else "SEM lags"))
elif "classical_lag" in dfs:
    winner_classical = "classical_lag"
else:
    winner_classical = "classical_nolag"

# MLP
if "mlp_nolag" in dfs and "mlp_lag" in dfs:
    w_nolag = mean_wis(dfs["mlp_nolag"])
    w_lag   = mean_wis(dfs["mlp_lag"])
    winner_mlp = "mlp_lag" if w_lag < w_nolag else "mlp_nolag"
    comparison.append(("MLP", w_nolag, w_lag,
                       "COM lags" if w_lag < w_nolag else "SEM lags"))
elif "mlp_lag" in dfs:
    winner_mlp = "mlp_lag"
else:
    winner_mlp = "mlp_nolag"

print("="*60)
print("COMPARAÇÃO LAGS vs SEM-LAGS (WIS médio — menor é melhor)")
print("="*60)
df_comp = pd.DataFrame(comparison, columns=["Família","WIS_sem_lag","WIS_com_lag","Vencedor"])
print(df_comp.to_string(index=False))
print()
print(f"→ Clássico escolhido para stacking : {winner_classical}")
print(f"→ MLP escolhido para stacking      : {winner_mlp}")
print(f"→ ESN (única versão)               : esn")

# Modelos-base finais do stacking
BASE_MODELS = {
    "Classical": winner_classical,
    "MLP"      : winner_mlp,
    "ESN"      : "esn",
}
print(f"\nModelos-base do stacking: {list(BASE_MODELS.values())}")

# Gráfico de barras da comparação
if comparison:
    fig, ax = plt.subplots(figsize=(8,4))
    x = np.arange(len(df_comp)); w = 0.35
    ax.bar(x-w/2, df_comp["WIS_sem_lag"], w, label="Sem lags", color="#4393c3")
    ax.bar(x+w/2, df_comp["WIS_com_lag"], w, label="Com lags", color="#d6604d")
    ax.set_xticks(x); ax.set_xticklabels(df_comp["Família"])
    ax.set_ylabel("WIS médio"); ax.set_title("Impacto dos Lags Climáticos no WIS",
                                              fontweight="bold")
    ax.legend(); ax.spines[["top","right"]].set_visible(False)
    for i,row in df_comp.iterrows():
        ax.text(i-w/2, row["WIS_sem_lag"], f"{row['WIS_sem_lag']:.1f}",
                ha="center", va="bottom", fontsize=8)
        ax.text(i+w/2, row["WIS_com_lag"], f"{row['WIS_com_lag']:.1f}",
                ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "comparacao_lags.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  Salvo: comparacao_lags.png")


classical_nolag    | média=inf | mediana_robusta=637.26
classical_lag      | média=7.55e+03 | mediana_robusta=393.25
COMPARAÇÃO LAGS vs SEM-LAGS (WIS médio — menor é melhor)
  Família  WIS_sem_lag  WIS_com_lag Vencedor
Clássicos   637.262869   393.253337 COM lags
      MLP   212.503599   229.386383 SEM lags

→ Clássico escolhido para stacking : classical_lag
→ MLP escolhido para stacking      : mlp_nolag
→ ESN (única versão)               : esn

Modelos-base do stacking: ['classical_lag', 'mlp_nolag', 'esn']
  Salvo: comparacao_lags.png


### 4. Tabela Unificada dos Modelos-Base

Funde os três modelos vencedores numa tabela longa, com WIS por (UF, validation, date, modelo).

In [63]:
# Para clássicos: o parquet pode conter VÁRIOS modelos (ETS/ARIMA/SARIMAX).
# Para o melhor-modelo e o stacking, primeiro reduzimos os clássicos ao
# melhor modelo clássico por (uf, validation) com base no WIS.
def collapse_classical(df):
    """Mantém, para cada (uf, validation, date), a previsão do modelo clássico
       com menor WIS médio naquele (uf, validation)."""
    if "model" not in df.columns:
        return df
    # WIS médio por uf×validation×model
    rank = (df.groupby(["uf","validation","model"])["WIS"]
              .mean().reset_index()
              .sort_values("WIS"))
    best = rank.drop_duplicates(["uf","validation"])[["uf","validation","model"]]
    best = best.rename(columns={"model":"best_model"})
    merged = df.merge(best, on=["uf","validation"])
    out = merged[merged["model"] == merged["best_model"]].copy()
    return out

base_frames = {}
for label, src in BASE_MODELS.items():
    d = dfs[src].copy()
    if label == "Classical":
        d = collapse_classical(d)
        d["model_family"] = "Classical"
    else:
        d["model_family"] = label
    base_frames[label] = d

# Tabela longa unificada
keep_cols = ["uf","uf_code","validation","date","y_true","median","WIS","model_family"] \
            + [f"lower_{int(a*100)}" for a in PI_LEVELS] \
            + [f"upper_{int(a*100)}" for a in PI_LEVELS]
long_base = pd.concat(
    [d[[c for c in keep_cols if c in d.columns]] for d in base_frames.values()],
    ignore_index=True
)
print(f"Tabela base unificada: {len(long_base):,} linhas")
print(f"Famílias: {long_base['model_family'].unique().tolist()}")


Tabela base unificada: 13,962 linhas
Famílias: ['Classical', 'MLP', 'ESN']


### 5. Heatmap do Melhor Modelo por UF × Validation Test

Cada célula mostra o modelo (Classical/MLP/ESN) com **menor WIS** naquela combinação UF×validation, com o WIS abaixo. Cores distintas por modelo. À direita de cada linha, o modelo que venceu na maioria das UFs daquela validation.

In [64]:
# WIS médio por uf × validation × família
wis_grid = (long_base.groupby(["validation","uf","model_family"])["WIS"]
                     .mean().reset_index())

# Melhor família por uf × validation
idx_best = wis_grid.groupby(["validation","uf"])["WIS"].idxmin()
best_grid = wis_grid.loc[idx_best].reset_index(drop=True)

validations = sorted(best_grid["validation"].unique())
ufs_all     = sorted(best_grid["uf"].unique())

# Cores por modelo
MODEL_COLORS = {
    "Classical": "#4393c3",   # azul
    "MLP"      : "#d6604d",   # vermelho
    "ESN"      : "#5aae61",   # verde
}
model_to_int = {m:i for i,m in enumerate(MODEL_COLORS)}

# Matriz de cores e anotações
n_rows, n_cols = len(validations), len(ufs_all)
color_mat = np.full((n_rows, n_cols), np.nan)
annot     = [["" for _ in range(n_cols)] for _ in range(n_rows)]
row_winner = []

for ri, val in enumerate(validations):
    sub = best_grid[best_grid["validation"]==val].set_index("uf")
    fam_counts = {}
    for ci, uf in enumerate(ufs_all):
        if uf in sub.index:
            fam = sub.loc[uf,"model_family"]
            wis = sub.loc[uf,"WIS"]
            color_mat[ri,ci] = model_to_int[fam]
            annot[ri][ci]    = f"{fam[:4]}\n({wis:.1f})"
            fam_counts[fam]  = fam_counts.get(fam,0)+1
    row_winner.append(max(fam_counts, key=fam_counts.get) if fam_counts else "-")

# Colormap discreto
from matplotlib.colors import ListedColormap
cmap = ListedColormap([MODEL_COLORS[m] for m in MODEL_COLORS])

fig, ax = plt.subplots(figsize=(max(16, n_cols*0.85), n_rows*1.4 + 1))
im = ax.imshow(color_mat, cmap=cmap, vmin=0, vmax=len(MODEL_COLORS)-1, aspect="auto")

ax.set_xticks(range(n_cols)); ax.set_xticklabels(ufs_all, fontsize=9)
ax.set_yticks(range(n_rows))
ax.set_yticklabels([f"Validation {v}" for v in validations], fontsize=10)

for ri in range(n_rows):
    for ci in range(n_cols):
        if annot[ri][ci]:
            ax.text(ci, ri, annot[ri][ci], ha="center", va="center",
                    fontsize=6.5, color="white", fontweight="bold")

# Coluna extra à direita: vencedor da validation
for ri, win in enumerate(row_winner):
    ax.text(n_cols + 0.1, ri, f"◄ {win}", ha="left", va="center",
            fontsize=11, fontweight="bold", color=MODEL_COLORS.get(win,"black"))

ax.set_xlim(-0.5, n_cols + 1.8)
legend_patches = [mpatches.Patch(color=c, label=m) for m,c in MODEL_COLORS.items()]
ax.legend(handles=legend_patches, loc="upper center",
          bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=10)
ax.set_title("Melhor Modelo por UF × Validation Test (menor WIS)\n"
             "À direita: modelo dominante na validation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(OUT_DIR / "heatmap_melhor_modelo.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Salvo: heatmap_melhor_modelo.png")

# Resumo textual
print("\nVencedor por Validation Test:")
for v, w in zip(validations, row_winner):
    print(f"  Validation {v}: {w}")
print("\nContagem global de vitórias por modelo (uf×validation):")
print(best_grid["model_family"].value_counts().to_string())


  Salvo: heatmap_melhor_modelo.png

Vencedor por Validation Test:
  Validation 1: ESN
  Validation 2: Classical
  Validation 3: ESN
  Validation 4: MLP

Contagem global de vitórias por modelo (uf×validation):
model_family
ESN          43
MLP          35
Classical    26


### 6. Stacking — Meta-Learner por Quantil

Para cada quantil (incluindo a mediana), treina-se um meta-learner (Ridge não-negativo) que combina as previsões dos 3 modelos-base. Treino com **validação cruzada out-of-fold** para evitar overfitting do meta-modelo. O alvo é o valor observado `y_true`.

O stacking é treinado de forma agregada (todas UFs/validations juntas) por quantil.

In [65]:
# ── Monta matriz wide: uma coluna por (modelo × quantil) ─────────────────────
def build_wide(long_base):
    """Pivota para wide: index=(uf,validation,date), colunas=(família, quantil)."""
    frames = []
    for fam in long_base["model_family"].unique():
        sub = long_base[long_base["model_family"]==fam].copy()
        cols = {}
        for q in QUANTILES:
            c = q_to_col(q)
            cols[f"{fam}__q{q}"] = sub[c].values
        wide_fam = pd.DataFrame(cols)
        wide_fam["uf"]         = sub["uf"].values
        wide_fam["validation"] = sub["validation"].values
        wide_fam["date"]       = sub["date"].values
        wide_fam["y_true"]     = sub["y_true"].values
        frames.append(wide_fam.set_index(["uf","validation","date","y_true"]))
    wide = pd.concat(frames, axis=1).reset_index()
    return wide

wide = build_wide(long_base)
wide = wide.dropna().reset_index(drop=True)
print(f"Matriz wide para stacking: {wide.shape}")

families = list(BASE_MODELS.keys())

# Pinball loss (quantile loss) — métrica de treino do meta-learner por quantil
def pinball(y, f, q):
    d = y - f
    return np.mean(np.maximum(q*d, (q-1)*d))

# ── Treina meta-learner (Gradient Boosting Quantílico) por quantil ───────────
# Cada quantil é otimizado DIRETAMENTE pela pinball loss (loss="quantile"),
# coerente com a avaliação por WIS do desafio. Diferente de um RF independente
# por quantil, o GB quantílico não colapsa lower==upper, pois aprende a forma
# da distribuição condicional em cada nível.
#
# Mantém o espírito do caretStack do orientador (ensemble de árvores como
# meta-learner via boosting), com K-Fold out-of-fold para evitar overfitting.


N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Grid do meta-GB (pequeno — espelha a lógica de tuning do caretStack)
GB_GRID = {
    "max_depth"    : [2, 3],
    "learning_rate": [0.05, 0.1],
}
GB_N_ESTIMATORS = 300

stack_pred       = {q: np.zeros(len(wide)) for q in QUANTILES}
meta_importances = {q: [] for q in QUANTILES}

y_all = wide["y_true"].values

def best_gb_for_quantile(Xq, yq, q):
    """Mini grid search interno via pinball loss (3-fold)."""
    def pinball_loss(y, f, alpha):
        d = y - f
        return np.mean(np.maximum(alpha*d, (alpha-1)*d))
    best_cfg, best_score = None, np.inf
    inner = KFold(n_splits=3, shuffle=True, random_state=0)
    for depth, lr in itertools.product(GB_GRID["max_depth"],
                                       GB_GRID["learning_rate"]):
        scores = []
        for tr_i, va_i in inner.split(Xq):
            gb = GradientBoostingRegressor(
                loss="quantile", alpha=q,
                n_estimators=GB_N_ESTIMATORS,
                max_depth=depth, learning_rate=lr,
                random_state=42,
            )
            gb.fit(Xq[tr_i], yq[tr_i])
            scores.append(pinball_loss(yq[va_i], gb.predict(Xq[va_i]), q))
        m = np.mean(scores)
        if m < best_score:
            best_score, best_cfg = m, {"max_depth": depth, "learning_rate": lr}
    return best_cfg

for q in QUANTILES:
    Xq  = wide[[f"{fam}__q{q}" for fam in families]].values
    cfg = best_gb_for_quantile(Xq, y_all, q)   # melhor config via pinball loss
    oof = np.zeros(len(wide))
    for tr_idx, va_idx in kf.split(Xq):
        gb = GradientBoostingRegressor(
            loss="quantile", alpha=q,
            n_estimators=GB_N_ESTIMATORS,
            max_depth=cfg["max_depth"],
            learning_rate=cfg["learning_rate"],
            random_state=42,
        )
        gb.fit(Xq[tr_idx], y_all[tr_idx])
        oof[va_idx] = gb.predict(Xq[va_idx])
        meta_importances[q].append(gb.feature_importances_)
    stack_pred[q] = np.clip(oof, 0, None)
    if abs(q-0.5) < 1e-9:
        print(f"  meta-GB (mediana) → depth={cfg['max_depth']}, "
              f"lr={cfg['learning_rate']}")

# Importância média dos modelos-base no meta-learner
print("\nImportância média do meta-GB por modelo (mediana, q=0.5):")
imp50 = np.mean(meta_importances[0.5], axis=0)
for fam, imp in zip(families, imp50):
    print(f"  {fam:10s}: {imp:.3f}")

Matriz wide para stacking: (4654, 31)
  meta-GB (mediana) → depth=3, lr=0.1

Importância média do meta-GB por modelo (mediana, q=0.5):
  Classical : 0.545
  MLP       : 0.125
  ESN       : 0.330


### 7. Previsões do Stacking + WIS Final

In [ ]:
# Monta dataframe de previsões do stacking
stack_df = wide[["uf","validation","date","y_true"]].copy()

uf_map   = long_base[["uf","uf_code"]].drop_duplicates()
stack_df = stack_df.merge(uf_map, on="uf", how="left")
stack_df["median"] = stack_pred[0.5]
for a in PI_LEVELS:
    stack_df[f"lower_{int(a*100)}"] = stack_pred[(1-a)/2]
    stack_df[f"upper_{int(a*100)}"] = stack_pred[1-(1-a)/2]

# Garante monotonicidade dos quantis (lower ≤ median ≤ upper)
q_cols_sorted = ([f"lower_{int(a*100)}" for a in sorted(PI_LEVELS, reverse=True)]
                 + ["median"]
                 + [f"upper_{int(a*100)}" for a in sorted(PI_LEVELS)])
arr = stack_df[q_cols_sorted].values
arr = np.maximum.accumulate(arr, axis=1)   # força ordem crescente
stack_df[q_cols_sorted] = arr

cols_order = ["uf","uf_code","validation","date","y_true",
              "median",
              "lower_50","upper_50","lower_80","upper_80",
              "lower_90","upper_90","lower_95","upper_95"]
stack_df = stack_df[cols_order]

# WIS do stacking
stack_df = add_wis_column(stack_df)

# ── WIS final por UF × Validation ────────────────────────────────────────────
wis_stack = (stack_df.groupby(["validation","uf"])["WIS"]
                     .mean().reset_index()
                     .pivot(index="validation", columns="uf", values="WIS"))

print("="*60)
print("WIS FINAL DO STACKING — por UF × Validation Test")
print("="*60)
print(wis_stack.round(2).to_string())

print("\nWIS médio do stacking por Validation:")
print(stack_df.groupby("validation")["WIS"].mean().round(2).to_string())

# ── Comparação final: stacking vs cada base ──────────────────────────────────
print("\n" + "="*60)
print("COMPARAÇÃO FINAL — WIS médio global")
print("="*60)
comp_final = {"Stacking": stack_df["WIS"].mean()}
for fam in families:
    sub = long_base[long_base["model_family"]==fam]
    comp_final[fam] = sub["WIS"].mean()
comp_final = pd.Series(comp_final).sort_values()
print(comp_final.round(3).to_string())
best_overall = comp_final.idxmin()
print(f"\n→ MELHOR MODELO GERAL: {best_overall} (WIS = {comp_final.min():.3f})")


WIS FINAL DO STACKING — por UF × Validation Test
uf              AC       AL      AM       AP       BA       CE       DF       GO      MA        MG       MS      MT      PA      PB      PE      PI        PR       RJ      RN      RO      RR       RS        SC     SE        SP      TO
validation                                                                                                                                                                                                                                
1           692.53  3888.66  105.42    57.26   659.14  4459.49  4180.81  2597.27  105.50   4978.51   854.60  539.54  140.42  277.73  323.50  322.00   9424.16  1177.20  449.14  154.02   94.45  5694.98   3150.91  58.25   8096.83  356.70
2           238.20   453.73  145.27   882.14  3176.29  1147.17  4587.58  3690.61  136.59  24149.18  1779.18  682.78  210.32  221.30  378.58  175.93  10720.20  4045.28  218.84  209.31   36.17  6330.23   8158.78  40.70  24014.56   95.08
3          

### 8. Export & Heatmap WIS do Stacking

In [ ]:
# Export
stack_df.to_parquet(OUT_DIR / "forecasts_stacking.parquet", index=False)
wis_stack.to_csv(OUT_DIR / "wis_stacking_uf_validation.csv")

# Salva importâncias do meta-RF (substitui meta_weights do Ridge)
imp_rows = []
for q in QUANTILES:
    imean = np.mean(meta_importances[q], axis=0)
    for fam, imp in zip(families, imean):
        imp_rows.append({"quantile": q, "model": fam, "importance": imp})
pd.DataFrame(imp_rows).to_csv(OUT_DIR / "meta_importances.csv", index=False)

# Heatmap WIS do stacking por UF × validation
fig, ax = plt.subplots(figsize=(max(14, wis_stack.shape[1]*0.7), 3.5))
im = ax.imshow(wis_stack.values, cmap="Reds",
               vmin=np.nanpercentile(wis_stack.values,5),
               vmax=np.nanpercentile(wis_stack.values,95), aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label="WIS")
ax.set_xticks(range(wis_stack.shape[1]))
ax.set_xticklabels(wis_stack.columns, fontsize=8)
ax.set_yticks(range(wis_stack.shape[0]))
ax.set_yticklabels([f"Val {v}" for v in wis_stack.index], fontsize=10)
vmax_ann = np.nanpercentile(wis_stack.values, 80)
for ri in range(wis_stack.shape[0]):
    for ci in range(wis_stack.shape[1]):
        v = wis_stack.values[ri,ci]
        if not np.isnan(v):
            ax.text(ci, ri, f"{v:.1f}", ha="center", va="center",
                    fontsize=7, color="white" if v>vmax_ann else "black")
ax.set_title("WIS do Stacking Ensemble — UF × Validation Test",
             fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(OUT_DIR / "heatmap_wis_stacking.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Salvo: forecasts_stacking.parquet, wis_stacking_uf_validation.csv,")
print("         meta_importances.csv, heatmap_wis_stacking.png")
print("\n✓ Stacking concluído.")


  Salvo: forecasts_stacking.parquet, wis_stacking_uf_validation.csv,
         meta_weights.csv, heatmap_wis_stacking.png

✓ Stacking concluído.


In [68]:
import matplotlib.dates as mdates
# ── Carrega série histórica para exibir treino nos plots ──────────────────────
hf_hist = pd.read_parquet(
    BASE_DIR / "0_databases" / "hierarch_forecast_lagged.parquet"
)
hf_hist = hf_hist[hf_hist["uf"] != "ES"]
hf_hist["date"]  = pd.to_datetime(hf_hist["date"])
hf_hist["casos"] = hf_hist["casos"].astype(float)

flag_cols_h  = ["train_1","train_2","train_3","train_4",
                "target_1","target_2","target_3","target_4"]
hf_hist_uf = (
    hf_hist.groupby(["uf","date"], sort=True)
           .agg({"casos":"sum",
                 **{c:"first" for c in flag_cols_h}})
           .reset_index()
)
for c in flag_cols_h:
    hf_hist_uf[c] = hf_hist_uf[c].astype(bool)

VALIDATIONS_MAP = [
    (1, "train_1", "target_1"),
    (2, "train_2", "target_2"),
    (3, "train_3", "target_3"),
    (4, "train_4", "target_4"),
]

# ── Mosaico do Stacking por Validation Test ───────────────────────────────────
pal = ["#f4a582","#d6604d","#b2182b","#67001f"]
ufs_sorted = sorted(stack_df["uf"].unique())
N = len(ufs_sorted); N_COLS = 6; N_ROWS = int(np.ceil(N / N_COLS))

for val_num, train_flag, target_flag in VALIDATIONS_MAP:

    df_v = stack_df[stack_df["validation"] == val_num].copy()
    if df_v.empty:
        continue

    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(28, N_ROWS*3.8),
                              constrained_layout=True)
    axes_flat = axes.flatten()

    for i, uf in enumerate(ufs_sorted):
        ax  = axes_flat[i]
        sub = df_v[df_v["uf"] == uf].sort_values("date")
        if sub.empty:
            ax.set_visible(False); continue

        hist = (hf_hist_uf[(hf_hist_uf["uf"] == uf) &
                            (hf_hist_uf[train_flag] == True)]
                .sort_values("date"))
        tgt  = (hf_hist_uf[(hf_hist_uf["uf"] == uf) &
                            (hf_hist_uf[target_flag] == True)]
                .sort_values("date"))

        ax.plot(hist["date"], hist["casos"], color="#2166ac", lw=0.7, label="Treino")
        ax.plot(tgt["date"],  tgt["casos"],  color="black",   lw=1.3, label="Observado")

        for j, a in enumerate(sorted(PI_LEVELS, reverse=True)):
            ax.fill_between(sub["date"],
                            sub[f"lower_{int(a*100)}"],
                            sub[f"upper_{int(a*100)}"],
                            alpha=0.22, color=pal[j])

        ax.plot(sub["date"], sub["median"], color="#d6604d", lw=1.2, ls="--")
        ax.set_title(f"{uf}  (WIS={sub['WIS'].mean():.1f})",
                     fontsize=8.5, fontweight="bold", pad=3)

        # Tick todo ano, rótulo a cada 2 anos (anos pares)
        ax.xaxis.set_major_locator(mdates.YearLocator(1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        for lbl in ax.get_xticklabels():
            try:    lbl.set_visible(int(lbl.get_text()) % 2 == 0)
            except: pass
        ax.tick_params(axis="x", labelsize=6.5, rotation=45)
        ax.tick_params(axis="y", labelsize=6.5)

        # Janela: 4 anos de treino + período target
        ax.set_xlim(hist["date"].max() - pd.DateOffset(years=4),
                    sub["date"].max())
        ax.set_ylim(bottom=0)
        ax.spines[["top","right"]].set_visible(False)

    for j in range(N, len(axes_flat)): axes_flat[j].set_visible(False)

    handles = [
        plt.Line2D([0],[0], color="#2166ac", lw=1.5, label="Treino"),
        plt.Line2D([0],[0], color="black",   lw=1.5, label="Observado"),
        plt.Line2D([0],[0], color="#d6604d", lw=1.5, ls="--", label="Mediana"),
    ] + [plt.Rectangle((0,0),1,1, color=pal[j], alpha=0.5,
                        label=f"IP {int(a*100)}%")
         for j,a in enumerate(sorted(PI_LEVELS, reverse=True))]
    fig.legend(handles=handles, loc="lower center",
               ncol=7, fontsize=9, bbox_to_anchor=(0.5,-0.015))
    fig.suptitle(f"Validation {val_num} — Stacking Ensemble | Casos por UF",
                 fontsize=13, fontweight="bold")
    fig.savefig(OUT_DIR / f"v{val_num}_stacking_uf_mosaic.png",
                dpi=140, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: v{val_num}_stacking_uf_mosaic.png")

  Salvo: v1_stacking_uf_mosaic.png
  Salvo: v2_stacking_uf_mosaic.png
  Salvo: v3_stacking_uf_mosaic.png
  Salvo: v4_stacking_uf_mosaic.png
